In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

# Load dataset
df = pd.read_csv("keystroke.csv")

# Keep only numeric columns (drop IDs like 's011')
df = df.select_dtypes(include=[np.number])

# Auto-detect last column as target
target_col = df.columns[-1]
print("Target column detected:", target_col)

X = df.iloc[:, :-1]   # all numeric features
y = df.iloc[:, -1]    # target column

# Convert target to discrete classes
# If continuous floats, threshold at median
if len(np.unique(y)) > 10:   # too many unique values → continuous
    y = (y > y.median()).astype(int)
else:
    y = y.astype(int)

# Check class distribution
print("Class distribution:\n", y.value_counts())

# Handle stratify issue
if y.value_counts().min() < 2:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
    print("⚠️ Stratify removed because one class has <2 samples")
else:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

# Scale
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


Target column detected: H.Return
Class distribution:
 H.Return
0    10202
1    10198
Name: count, dtype: int64


In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

param_grid = {
    "logistic": {"C":[0.1,1,10], "solver":["liblinear","lbfgs"]},
    "knn": {"n_neighbors":[3,5,7], "weights":["uniform","distance"]},
    "svm": {"C":[0.1,1,10], "kernel":["linear","rbf"], "probability":[True]},
    "decision_tree": {"max_depth":[None,5,10], "criterion":["gini","entropy"]},
    "random_forest": {"n_estimators":[50,100], "max_depth":[None,10]}
}

models = {
    "logistic": LogisticRegression(max_iter=1000),
    "knn": KNeighborsClassifier(),
    "svm": SVC(),
    "decision_tree": DecisionTreeClassifier(),
    "random_forest": RandomForestClassifier()
}


In [6]:
best_models = {}
metrics_list = []

for name, model in models.items():
    grid = GridSearchCV(model, param_grid[name], cv=5, scoring='accuracy')
    grid.fit(X_train_scaled, y_train)
    best_models[name] = grid.best_estimator_
    
    y_pred = grid.best_estimator_.predict(X_test_scaled)
    metrics_list.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, average='weighted', zero_division=0),
        "Recall": recall_score(y_test, y_pred, average='weighted', zero_division=0),
        "F1": f1_score(y_test, y_pred, average='weighted', zero_division=0)
    })


In [7]:
metrics_df = pd.DataFrame(metrics_list)
metrics_df.to_csv("model_comparison.csv", index=False)
print(metrics_df)


           Model  Accuracy  Precision    Recall        F1
0       logistic  0.701961   0.701966  0.701961  0.701959
1            knn  0.793627   0.793700  0.793627  0.793615
2            svm  0.781618   0.782085  0.781618  0.781527
3  decision_tree  0.732108   0.732109  0.732108  0.732107
4  random_forest  0.801471   0.801890  0.801471  0.801402


In [8]:
from sklearn.ensemble import VotingClassifier
import pickle

voting_clf = VotingClassifier(
    estimators=[(name, best_models[name]) for name in best_models],
    voting='soft'
)
voting_clf.fit(X_train_scaled, y_train)

y_pred_voting = voting_clf.predict(X_test_scaled)
print("Voting Classifier Accuracy:", accuracy_score(y_test, y_pred_voting))
print("Voting Classifier Report:\n", classification_report(y_test, y_pred_voting))

# Save scaler and voting classifier
pickle.dump(scaler, open("scaler.pkl","wb"))
pickle.dump(voting_clf, open("voting_clf.pkl","wb"))


Voting Classifier Accuracy: 0.7985294117647059
Voting Classifier Report:
               precision    recall  f1-score   support

           0       0.81      0.78      0.79      2040
           1       0.79      0.82      0.80      2040

    accuracy                           0.80      4080
   macro avg       0.80      0.80      0.80      4080
weighted avg       0.80      0.80      0.80      4080



In [10]:
import pickle
import numpy as np

# Load saved scaler and voting classifier
scaler = pickle.load(open("scaler.pkl","rb"))
voting_clf = pickle.load(open("voting_clf.pkl","rb"))

# Take user input
print("Enter feature values separated by space:")
user_input = list(map(float, input().split()))

# Preprocess
X_new = np.array(user_input).reshape(1,-1)
X_new_scaled = scaler.transform(X_new)

# Predict
prediction = voting_clf.predict(X_new_scaled)
print("Predicted Label:", prediction[0])


Enter feature values separated by space:


 0.12 0.08 0.15 0.09 0.11 0.10 0.14 0.13 0.12 0.09 0.10 0.11 0.13 0.12 0.14 0.15 0.16 0.12 0.11 0.09 0.10 0.13 0.12 0.14 0.15 0.16 0.12 0.11 0.09 0.10 0.13 0.12


Predicted Label: 1


C:\ProgramData\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
